# Galaxy Stellar Mass Estimation using CNN (SDSS DR17 + FIREFLY)
This notebook downloads a FIREFLY (DR17) subset, retrieves SDSS cutouts, preprocesses images, trains a CNN (ResNet50 transfer learning) to predict stellar mass (log(M\*/M☉)), and evaluates & saves results.

**Notes before running:**
- This notebook may download large files (FIREFLY catalog). Run in Colab or a machine with enough disk/compute.
- If running in Colab, enable GPU (Runtime → Change runtime type → GPU).


In [ ]:
# 0. Install required packages (uncomment if needed)
#!pip install astroquery astropy tensorflow pandas matplotlib scikit-learn pillow requests


## 1. Imports and folder setup

In [ ]:
import os
import requests
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import asyncio
import aiohttp
from tqdm import tqdm
import nest_asyncio
from astropy.io import fits
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from tensorflow.keras.applications.resnet import preprocess_input, ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing import image


In [ ]:
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/raw/images', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
print('Folders created.')


## 2. Download FIREFLY DR17 catalog (or use local copy)
The FIREFLY full catalog is large; here we attempt to download the FITS file from SDSS SAS. If you already have a local copy, skip the download and set `firefly_path` to your file.

In [ ]:
firefly_path = "/content/data/raw/manga-firefly-globalprop-v3_1_1-mastar.fits"

## 3. Read FIREFLY and assemble a manageable sample
We load the FITS table, filter for valid stellar mass entries and select a random sample to keep runtime reasonable for prototyping.

In [ ]:
print('Reading FIREFLY FITS...')
try:
    hdul = fits.open(firefly_path)
    data=hdul[1].data
    dtype_native = np.dtype([
        (name, data[name].dtype.newbyteorder('='))
        for name in data.names
    ])
    data_native = np.array(data, dtype=dtype_native)
    #Create DataFrame directly from FITS data and handle byte order
    df_firefly = pd.DataFrame(data_native)

except FileNotFoundError:
     print(f"Error: FIREFLY file not found at {firefly_path}. Please ensure the file was downloaded correctly in the previous step.")
     raise # Re-raise the error
except Exception as e:
    print(f"An error occurred while reading the FITS file: {e}")
    raise # Re-raise other exceptions


print('Total FIREFLY entries:', len(df_firefly))

# Select galaxies with positive stellar mass
mass_col = "PHOTOMETRIC_MASS"

# Filter for valid mass entries
if mass_col in df_firefly.columns:
    df = df_firefly[df_firefly[mass_col].notna() & (df_firefly[mass_col] > 0)].copy()
    df = df.reset_index(drop=True)
    print('Entries with positive mass:', len(df))

    # For prototyping, sample up to N galaxies
    N = 10000
    if len(df) > N:
        df = df.sample(N, random_state=42).reset_index(drop=True)
    print('Using sample size:', len(df))
else:
    print(f"Error: Mass column '{mass_col}' not found in the DataFrame.")
    df = pd.DataFrame() # Create an empty DataFrame to avoid errors in subsequent steps

In [ ]:
#Check the column names to select RA, DEC and Mass columns
df.columns

## 4. Download SDSS image cutouts (gri composite via SkyServer JPEG API)
We fetch a square JPEG centered on each galaxy using RA/Dec. Adjust `scale` and `size` if needed.

In [ ]:
nest_asyncio.apply()  # allows nested event loops

# --- Parameters ---
image_dir = "data/raw/images"
os.makedirs(image_dir, exist_ok=True)
mass_col = "PHOTOMETRIC_MASS"
ra_col, dec_col = "OBJRA", "OBJDEC"
scale, size = 0.396, 128
n_images = 10000  # number of images to download

# --- Helper: build URL ---
def make_sdss_url(ra, dec, scale=0.396, size=128):
    return (
        f"https://skyserver.sdss.org/dr17/SkyServerWS/ImgCutout/getjpeg?"
        f"ra={ra}&dec={dec}&scale={scale}&width={size}&height={size}"
    )

# --- Async downloader ---
async def fetch_and_save(session, idx, ra, dec, mass, image_dir):
    url = make_sdss_url(ra, dec, scale, size)
    filename = os.path.join(image_dir, f"gal_{idx}.jpg")

    # Skip if file already exists
    if os.path.exists(filename):
        return filename, mass

    try:
        async with session.get(url, timeout=120) as r:
            if r.status == 200:
                content = await r.read()
                with open(filename, "wb") as f:
                    f.write(content)
                return filename, mass
    except Exception:
        pass  # network timeout or missing coverage

    return None, None

# --- Main async function ---
async def download_all(df, limit=n_images, batch_size=100):
    connector = aiohttp.TCPConnector(limit_per_host=batch_size)
    timeout = aiohttp.ClientTimeout(total=60)
    img_paths, masses = [], []

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        tasks = []
        for idx, row in df.iloc[:limit].iterrows():
            ra, dec, mass = row[ra_col], row[dec_col], row[mass_col]
            if pd.isna(ra) or pd.isna(dec) or pd.isna(mass) or mass <= 0:
                img_paths.append(None)
                masses.append(None)
                continue

            tasks.append(fetch_and_save(session, idx, ra, dec, mass, image_dir))

        results = []
        for f in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Downloading SDSS cutouts"):
            result = await f
            results.append(result)

    # Unpack results
    for p, m in results:
        img_paths.append(p)
        masses.append(m)

    return img_paths, masses

# --- Run the async download ---
img_paths, mass_list = await download_all(df, limit=10000, batch_size=100)

# --- Add to DataFrame and save ---
df['image_path'] = img_paths
df['mass'] = mass_list

print("Images downloaded:", sum(p is not None for p in img_paths))
df.to_csv("data/processed/galaxy_mass_catalog.csv", index=False)


## 5. Quick visual quality check (plot random sample)

In [ ]:
import random
from PIL import Image
valid_imgs = df[df['image_path'].notnull()].reset_index(drop=True)
sample = valid_imgs.sample(min(25, len(valid_imgs)), random_state=1)
plt.figure(figsize=(10,10))
for i, row in enumerate(sample.itertuples()):
    try:
        img = Image.open(row.image_path)
        plt.subplot(5,5,i+1)
        plt.imshow(img)
        plt.title(f"Mass={row.mass:.2f}")
        plt.axis('off')
    except Exception as e:
        pass
plt.tight_layout()


## 6. Preprocess images: load, resize to 128*128 (already) and filter invalid entries

In [ ]:
def load_image_array(path):
    try:
        img = image.load_img(path, target_size=(128,128))
        arr = image.img_to_array(img)
        return arr
    except Exception:
        return None

rows = df[df['image_path'].notnull()].copy()
X_list, y_list = [], []
for r in rows.itertuples():
    arr = load_image_array(r.image_path)
    if arr is None:
        continue
    if np.isfinite(r.mass):
        X_list.append(arr)
        y_list.append(r.mass)

X = np.array(X_list)
y = np.array(y_list)
print('Final dataset shape:', X.shape, y.shape)
np.savez('data/processed/processed_dataset.npz', X=X, y=y)


## 7. Train/Validation/Test split (60/20/20)

In [ ]:
# 7. Train/Validation/Test split (60/20/20)
data_loaded = np.load('data/processed/processed_dataset.npz')
X = data_loaded['X']
y = data_loaded['y']

# First split: separate test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Second split: divide temp into train (60% of total) and validation (20% of total)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

print(f"Train: {X_train.shape} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation: {X_val.shape} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test: {X_test.shape} ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nMass range - Train: [{y_train.min():.2f}, {y_train.max():.2f}]")
print(f"Mass range - Val: [{y_val.min():.2f}, {y_val.max():.2f}]")
print(f"Mass range - Test: [{y_test.min():.2f}, {y_test.max():.2f}]")


In [ ]:
#Download train, validation and test datasets
# Create directory to save splits
os.makedirs('data/splits', exist_ok=True)

# Save each split as a compressed NPZ file
np.savez_compressed('data/splits/train.npz', X=X_train, y=y_train)
np.savez_compressed('data/splits/val.npz', X=X_val, y=y_val)
np.savez_compressed('data/splits/test.npz', X=X_test, y=y_test)

print("✅ Saved datasets:")
print("- data/splits/train.npz")
print("- data/splits/val.npz")
print("- data/splits/test.npz")

import shutil

# Zip the folder containing all splits
shutil.make_archive('dataset_splits', 'zip', 'data/splits')

# For Colab: download the zip file
from google.colab import files
files.download('dataset_splits.zip')


## 8. Data augmentation and preprocessing for ResNet
We will use ImageDataGenerator for augmentation and ResNet50 preprocess_input.

In [ ]:
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=50,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.25,
    brightness_range=[0.8, 1.2],
    shear_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_gen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

# Create data flows
train_flow = train_gen.flow(X_train, y_train, batch_size=32, shuffle=True)
val_flow = val_gen.flow(X_val, y_val, batch_size=32, shuffle=False)
test_flow = test_gen.flow(X_test, y_test, batch_size=32, shuffle=False)

print(f"Train batches per epoch: {len(train_flow)}")
print(f"Validation batches per epoch: {len(val_flow)}")
print(f"Test batches: {len(test_flow)}")



## 9. Build ResNet50-based regression model

In [ ]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(128,128,3))
for layer in base_model.layers[-30:]:
    layer.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(256, activation='relu', kernel_regularizer='l2')(x)
x= Dropout(0.5)(x)
x= Dense(64, activation='relu', kernel_regularizer='l2')(x)
x= Dropout(0.3)(x)
output = Dense(1, activation='linear', kernel_regularizer='l2')(x)
model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer=Adam(1e-4), loss='mse', metrics=['mae'])
model.summary()

## 10. Train the model

In [ ]:
callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    EarlyStopping(monitor='val_loss',patience=7, restore_best_weights=True),
    ModelCheckpoint('models/cnn_finetuned_model1.h5', save_best_only=True)
]
history = model.fit(
    train_flow,
    validation_data=val_flow,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)


## 11. Evaluate & visualize results

In [ ]:
# Reset generator for consistent order
test_flow.reset()

# Compute exact number of steps
steps = math.ceil(test_flow.n / test_flow.batch_size)

# Run prediction
y_pred = model.predict(
    test_flow,
    steps=steps,
    verbose=1
).ravel()

# True labels (same order since shuffle=False)
y_true = y_test[:len(y_pred)]

# After model evaluation
results = {
    'Metric': ['R² Score', 'MAE (dex)', 'RMSE (dex)', 'Median Absolute Error (dex)'],
    'Test Set': [{r2_score(y_true, y_pred):.3f}, {mean_absolute_error(y_true, y_pred):.3f}, {np.sqrt(mean_squared_error(y_true, y_pred)):.3f}, np.median(np.abs(y_test - y_pred))]
}

results_df = pd.DataFrame(results)
print(results_df.to_markdown(index=False))  # For reports


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss plot
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss (MSE)', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# MAE plot
axes[1].plot(history.history['mae'], label='Training MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('MAE (dex)', fontsize=12)
axes[1].set_title('Training and Validation MAE', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# Scatter plot with transparency
ax.scatter(y_test, y_pred, alpha=0.3, s=20, edgecolors='none')

# Perfect prediction line (1:1 line)
min_val, max_val = 7, 12
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

# Formatting
ax.set_xlabel('True log(M*/M☉)', fontsize=14, fontweight='bold')
ax.set_ylabel('Predicted log(M*/M☉)', fontsize=14, fontweight='bold')
ax.set_title(f'Galaxy Stellar Mass Predictions (R² = {r2_score(y_true, y_pred):.3f})', fontsize=16, fontweight='bold')
ax.set_xlim(7, 12)
ax.set_ylim(7, 12)
ax.legend(fontsize=12)
ax.grid(alpha=0.3)

# Add text box with metrics
textstr = f'MAE = {mean_absolute_error(y_true, y_pred):.3f} dex\nRMSE = {np.sqrt(mean_squared_error(y_true, y_pred)):.3f} dex\nN = {len(y_test)}'
ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.savefig('predicted_vs_actual.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
residuals = y_test - y_pred

fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(y_pred, residuals, alpha=0.3, s=20, edgecolors='none')
ax.axhline(y=0, color='r', linestyle='--', linewidth=2, label='Zero Error')

# Add ±0.3 dex lines (typical systematic uncertainty)
ax.axhline(y=0.3, color='orange', linestyle=':', linewidth=1.5, alpha=0.7, label='±0.3 dex')
ax.axhline(y=-0.3, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)

ax.set_xlabel('Predicted log(M*/M☉)', fontsize=14, fontweight='bold')
ax.set_ylabel('Residual (True - Predicted) [dex]', fontsize=14, fontweight='bold')
ax.set_title('Residual Plot', fontsize=16, fontweight='bold')
ax.set_xlim(7, 12)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.savefig('residual_plot.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
plt.violinplot([y_test, y_pred], showmeans=True)
plt.xticks([1, 2], ['True Mass', 'Predicted Mass'])
plt.ylabel('Stellar Mass')
plt.title('Distribution of True and Predicted Stellar Masses')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.hist(residuals, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(x=0, color='r', linestyle='--', linewidth=2, label='Zero Error')
ax.axvline(x=np.median(residuals), color='green', linestyle='--', linewidth=2,
           label=f'Median = {np.median(residuals):.3f} dex')

ax.set_xlabel('Residual (True - Predicted) [dex]', fontsize=14, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=14, fontweight='bold')
ax.set_title('Distribution of Prediction Errors', fontsize=16, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3, axis='y')

plt.savefig('error_distribution.png', dpi=300, bbox_inches='tight')
plt.show()